# Homework: The EM Algorithm

## Problem 1: EM for a Two-Component Exponential Mixture

Suppose you observe $n$ data points $y_1, \ldots, y_n$ from a two-component exponential mixture:

$$f(y_i | \boldsymbol{\theta}) = \pi \, \lambda_1 e^{-\lambda_1 y_i} + (1 - \pi) \, \lambda_2 e^{-\lambda_2 y_i}, \quad y_i > 0$$

where $\boldsymbol{\theta} = (\pi, \lambda_1, \lambda_2)$.

**(a)** Write down the complete-data log-likelihood, treating the component membership $z_i \in \{1, 2\}$ as the latent variable.

**(b)** Derive the E-step: compute the responsibility $\gamma_i^{(t)} = P(z_i = 1 | y_i, \boldsymbol{\theta}^{(t)})$.

**(c)** Derive the M-step: write the update formulas for $\pi^{(t+1)}$, $\lambda_1^{(t+1)}$, and $\lambda_2^{(t+1)}$.

**(d)** Implement the EM algorithm for this model. Your function should have the following signature and return a dictionary containing the estimated parameters and the log-likelihood history.

In [ ]:
import numpy as np
from scipy import stats

def em_exponential_mixture(y, pi_init, lam1_init, lam2_init,
                           max_iter=200, tol=1e-8):
    """EM algorithm for a 2-component exponential mixture.

    Parameters
    ----------
    y : np.ndarray
        Observed data of shape (n,), all positive.
    pi_init : float
        Initial mixing proportion for component 1.
    lam1_init, lam2_init : float
        Initial rate parameters.
    max_iter : int
        Maximum number of iterations.
    tol : float
        Convergence tolerance on log-likelihood change.

    Returns
    -------
    dict with keys: pi, lam1, lam2, loglik_history
    """
    # Your code here
    pass

Test your implementation on the following simulated data and report the estimated parameters:

In [ ]:
np.random.seed(123)
n = 500
pi_true = 0.35
lam1_true, lam2_true = 0.5, 3.0
z = np.random.binomial(1, 1 - pi_true, n)
y = np.where(z == 0,
             np.random.exponential(1 / lam1_true, n),
             np.random.exponential(1 / lam2_true, n))

result = em_exponential_mixture(y, pi_init=0.5, lam1_init=0.3, lam2_init=2.0)

## Problem 2: Diagnosing EM Convergence

A colleague provides the following EM implementation for the two-component Gaussian mixture model. Read the code carefully and answer the questions below.

In [ ]:
import numpy as np
from scipy import stats

def em_gmm_v2(y, pi_init, mu1_init, sigma1_init, mu2_init, sigma2_init,
              max_iter=200, tol=1e-8):
    n = len(y)
    pi = pi_init
    mu1, mu2 = mu1_init, mu2_init
    s1, s2 = sigma1_init, sigma2_init
    loglik_history = []

    for iteration in range(max_iter):
        d1 = pi * stats.norm.pdf(y, mu1, s1)
        d2 = (1 - pi) * stats.norm.pdf(y, mu2, s2)
        gamma = d1 / (d1 + d2)

        ll = np.sum(np.log(d1 + d2))
        loglik_history.append(ll)

        if iteration > 0 and loglik_history[-1] - loglik_history[-2] < tol:
            break

        n1 = np.sum(gamma)
        n2 = n - n1

        pi = n1 / n
        mu1 = np.sum(gamma * y) / n1
        mu2 = np.sum((1 - gamma) * y) / n2
        s1 = np.sqrt(np.sum(gamma * (y - mu1) ** 2) / n1)
        s2 = np.sqrt(np.sum((1 - gamma) * (y - mu2) ** 2) / n2)

    return {
        "pi": pi, "mu1": mu1, "sigma1": s1,
        "mu2": mu2, "sigma2": s2,
        "loglik_history": loglik_history,
    }

**(a)** There is a subtle bug in the convergence check. Identify it and explain what incorrect behavior it could cause.

**(b)** Suppose this function is called with `sigma1_init=0.001` on a dataset where one observation happens to be very close to `mu1_init`. Explain what numerical issue could arise during the E-step and how you would fix it.

**(c)** Does this implementation handle the label-switching symmetry of the mixture model? That is, if you swap the roles of component 1 and component 2 in the initialization, do you get an equivalent solution? Explain why or why not.

## Problem 3: EM for Missing Data

Consider a dataset of $n = 200$ paired measurements $(X_i, Y_i)$ from a bivariate normal distribution, where 25% of $Y$ values are missing at random.

**(a)** Write a function that implements the EM algorithm for estimating the bivariate normal parameters $(\mu_X, \mu_Y, \sigma_X^2, \sigma_Y^2, \rho)$ when $Y$ has missing values. Use the following signature:

In [ ]:
def em_bivariate(X, Y, observed, max_iter=200, tol=1e-8):
    """EM for bivariate normal with missing Y values.

    Parameters
    ----------
    X : np.ndarray
        Fully observed variable, shape (n,).
    Y : np.ndarray
        Partially observed variable, shape (n,). NaN for missing.
    observed : np.ndarray
        Boolean array, True where Y is observed.
    max_iter : int
        Maximum iterations.
    tol : float
        Convergence tolerance on max parameter change.

    Returns
    -------
    dict with keys: mu_x, mu_y, var_x, var_y, rho, n_iter
    """
    # Your code here
    pass

**(b)** Test your implementation on the following simulated data. Compare the EM estimates with the complete-case estimates (using only observations where $Y$ is observed) and the full-data estimates (using the true $Y$ values before deletion).

In [ ]:
np.random.seed(42)
n = 200
mu = np.array([3.0, 7.0])
rho_true = 0.6
sx, sy = 1.5, 2.0
Sigma = np.array([[sx**2, rho_true * sx * sy],
                   [rho_true * sx * sy, sy**2]])

data = np.random.multivariate_normal(mu, Sigma, n)
X, Y_full = data[:, 0], data[:, 1]

# Introduce 25% MAR missingness
prob_miss = 0.25 * np.ones(n)
missing = np.random.binomial(1, prob_miss, n).astype(bool)
Y = Y_full.copy()
Y[missing] = np.nan
observed = ~missing

**(c)** Which estimator do you expect to have smaller variance for $\rho$, the EM estimator or the complete-case estimator? Explain why.

## Problem 4: Zero-Inflated Poisson Model Selection

The following code fits a standard Poisson model and a zero-inflated Poisson (ZIP) model to a dataset. Read the code and answer the questions.

In [ ]:
import numpy as np
from scipy import stats

def fit_poisson(y):
    """Fit a standard Poisson model by MLE."""
    lam_hat = np.mean(y)
    ll = np.sum(stats.poisson.logpmf(y, lam_hat))
    return {"lam": lam_hat, "loglik": ll, "n_params": 1}

def fit_zip_em(y, max_iter=200, tol=1e-8):
    """Fit a zero-inflated Poisson model by EM."""
    n = len(y)
    pi = 0.3
    lam = np.mean(y[y > 0])
    is_zero = (y == 0)
    loglik_history = []

    for iteration in range(max_iter):
        gamma = np.zeros(n)
        gamma[is_zero] = pi / (pi + (1 - pi) * np.exp(-lam))

        ll = (np.sum(is_zero) * np.log(pi + (1 - pi) * np.exp(-lam))
              + np.sum(~is_zero * (np.log(1 - pi)
                                    + stats.poisson.logpmf(y, lam))))
        loglik_history.append(ll)

        if iteration > 0 and abs(loglik_history[-1] - loglik_history[-2]) < tol:
            break

        pi = np.mean(gamma)
        lam = np.sum((1 - gamma) * y) / np.sum(1 - gamma)

    return {"pi": pi, "lam": lam, "loglik": ll,
            "n_params": 2, "loglik_history": loglik_history}

**(a)** In the E-step, why is $\gamma_i = 0$ for all observations with $y_i > 0$? Explain using the structure of the ZIP model.

**(b)** Given that the Poisson model is nested within the ZIP model (setting $\pi = 0$ recovers the Poisson), a natural model comparison tool is the likelihood ratio test. However, the standard chi-squared approximation for the LRT is not valid here. Explain why, in terms of the parameter space and the null hypothesis.

**(c)** Using the AIC ($\text{AIC} = -2\ell + 2k$, where $k$ is the number of parameters), compare the Poisson and ZIP fits on the following data. Which model does AIC prefer?

In [ ]:
np.random.seed(10)
n = 300
pi_true = 0.2
lam_true = 2.0
z = np.random.binomial(1, pi_true, n)
y = np.where(z == 1, 0, np.random.poisson(lam_true, n))

poisson_fit = fit_poisson(y)
zip_fit = fit_zip_em(y)

## Problem 5: EM with Asymmetric Components

Consider a two-component mixture where component 1 is $\text{Exponential}(\lambda)$ (supported on $[0, \infty)$) and component 2 is $N(\mu, \sigma^2)$ (supported on $(-\infty, \infty)$):

$$f(y_i | \boldsymbol{\theta}) = \pi \, \lambda e^{-\lambda y_i} \mathbb{1}(y_i > 0) + (1 - \pi) \, \phi(y_i | \mu, \sigma^2)$$

**(a)** Derive the E-step and M-step for this model. Pay careful attention to how the exponential component's support restriction affects the responsibility computation for observations with $y_i \leq 0$.

**(b)** Implement the EM algorithm for this model:

In [ ]:
def em_exp_normal_mixture(y, pi_init, lam_init, mu_init, sigma_init,
                          max_iter=200, tol=1e-8):
    """EM for exponential-normal mixture.

    Parameters
    ----------
    y : np.ndarray
        Observed data of shape (n,).
    pi_init : float
        Initial mixing proportion for the exponential component.
    lam_init : float
        Initial rate for the exponential component.
    mu_init : float
        Initial mean for the normal component.
    sigma_init : float
        Initial std dev for the normal component.
    max_iter : int
        Maximum iterations.
    tol : float
        Convergence tolerance on log-likelihood change.

    Returns
    -------
    dict with keys: pi, lam, mu, sigma, loglik_history
    """
    # Your code here
    pass

**(c)** Test on the following data and report your estimates:

In [ ]:
np.random.seed(42)
n = 400
pi_true = 0.4
lam_true = 2.0
mu_true, sigma_true = -1.0, 1.5
z = np.random.binomial(1, pi_true, n)
y = np.where(z == 1,
             np.random.exponential(1 / lam_true, n),
             np.random.normal(mu_true, sigma_true, n))

result = em_exp_normal_mixture(y, pi_init=0.5, lam_init=1.0,
                                mu_init=0.0, sigma_init=2.0)